# HOUSEVOICE openWakeWord trainer — CPU preparation + GPU training

This notebook trains the `HOUSEVOICE` OpenWakeWord model without changing Home Assistant. It deliberately separates long, resumable CPU preparation from the short CUDA-only training phase so dataset downloads and preprocessing do not consume Colab GPU quota.

## Required runtime

Use Colab runtime version **2026.07** for both runtime types:

- Python `3.12.13`
- NumPy `2.0.2`
- PyTorch `2.11.0`
- Torchaudio `2.11.0`
- At least **9 GB free in Google Drive** and **12 GB free in Colab local storage**

Do not install replacement Torch, Torchaudio, Torchvision, standalone CUDA, or `piper-phonemize-cross` packages.

## Exact two-phase workflow

### Phase 1 — CPU preparation (Steps 0–8A)

1. Select **Runtime → Change runtime type → Runtime Version 2026.07 → Hardware accelerator: None**.
2. Leave `RUNTIME_PHASE = 'CPU'` in Step 0.
3. For the current over-quota workspace only, run optional Step **0M** after Step 0; it derives the required 1.9008 GB subsets in local scratch before you remove legacy raw Drive files.
4. Run Steps **1 through 8A** in order.
5. Step 4 still requires you to listen and type `HOUSEVOICE`.
6. Wait until Step 8A reports that all four persistent feature files are verified.

CPU phase performs Drive mounting, dependency installation, source/voice restoration, speech validation, local-scratch raw downloads, audio conversion, TTS clip generation, augmentation, and feature extraction. Only compact prepared data is persisted to Drive. It must not hold a T4/L4 allocation.

### Phase 2 — GPU training (Step 8B only)

1. Disconnect the CPU runtime.
2. Select runtime **2026.07** with a **T4 or L4 GPU**.
3. Change only Step 0 to `RUNTIME_PHASE = 'GPU'`.
4. Rerun Steps **0 and 1 only**. Step 1 installs only the minimal GPU-training dependency set in this phase.
5. Jump directly to **Step 8B**. Do not rerun Steps 2–8A.
6. Step 8B validates the persistent config/features, stages prepared arrays sequentially into local `/content` scratch to avoid random Drive I/O, and resumes the Drive checkpoint if present.

### Phase 3 — CPU export (Step 9)

1. After Step 8B reports training complete, disconnect the GPU runtime immediately.
2. Return to runtime **2026.07** with **Hardware accelerator: None**.
3. Change Step 0 back to `RUNTIME_PHASE = 'CPU'`.
4. Rerun Steps **0 and 1 only**, then jump directly to **Step 9**.
5. Step 9 reloads the completed checkpoint and persistent test features, exports ONNX/TFLite on CPU, verifies them, and stores them in Drive.

## Recovery

- CPU interruption: reconnect with `RUNTIME_PHASE = 'CPU'` and rerun from Step 0 through the interrupted CPU step. Prepared Drive assets are skipped. Local ACAV/FMA raw downloads are lost with the VM and may repeat; ACAV fetches only missing subset ranges, while an incomplete FMA preparation requires the local archive download again.
- GPU interruption: reconnect with `RUNTIME_PHASE = 'GPU'`, rerun Steps 0 and 1, then Step 8B. Training resumes from the last atomic 250-step checkpoint.
- Export interruption: reconnect on CPU, rerun Steps 0 and 1, then Step 9.

Pinned sources remain OpenWakeWord `368c03716d1e92591906a84949bc477f3a834455` and Piper sample generator `2971426a55072f7d22fec416ca7800df8bd23207`. This revision is statically validated but has not been executed in Colab by the Home Assistant agent.


## 0. Choose CPU/GPU phase, mount Google Drive, and initialize persistence

Use `CPU` for Steps 0–8A and Step 9. Use `GPU` only for Step 8B. The cell fails if the selected phase does not match the attached hardware.


In [ ]:
import hashlib, json, os, re, time
from pathlib import Path

import requests
from google.colab import drive
from tqdm.auto import tqdm

RUNTIME_PHASE = 'CPU'  # Change to 'GPU' only for Step 8B.

TARGET_PHRASE = ['house voice']
MODEL_NAME    = 'housevoice'
LANGUAGE      = 'en_US'

assert RUNTIME_PHASE in ('CPU', 'GPU'), RUNTIME_PHASE

import torch
if RUNTIME_PHASE == 'CPU' and torch.cuda.is_available():
    raise RuntimeError(
        'CPU phase selected while a GPU is attached. Change the Colab hardware accelerator to None '
        'so preparation/export does not consume GPU quota.')
if RUNTIME_PHASE == 'GPU' and not torch.cuda.is_available():
    raise RuntimeError(
        'GPU phase selected but CUDA is unavailable. Attach a T4 or L4 before running Step 8B.')
print(f'Runtime phase: {RUNTIME_PHASE}; CUDA available: {torch.cuda.is_available()}')


def require_phase(expected, step_name):
    if RUNTIME_PHASE != expected:
        raise RuntimeError(
            f'{step_name} requires RUNTIME_PHASE={expected!r}; current phase is {RUNTIME_PHASE!r}.')


assert re.match(r'^[a-z]{2}_[A-Z]{2}$', LANGUAGE), \
    f'LANGUAGE must be a Piper locale like en_US or es_ES, got: {LANGUAGE!r}'
assert re.match(r'^[a-z0-9_-]+$', MODEL_NAME), MODEL_NAME

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive/HOUSEVOICE_Wakeword')
CACHE_ROOT = Path('/content/housevoice-cache')
LOCAL_DATA_ROOT = Path('/content/housevoice-raw')
SOURCE_ROOT = DRIVE_ROOT / 'sources'
MODEL_ROOT = DRIVE_ROOT / 'models'
DATA_ROOT = DRIVE_ROOT / 'datasets'
RUNS_ROOT = DRIVE_ROOT / 'runs'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
for directory in (CACHE_ROOT, LOCAL_DATA_ROOT, SOURCE_ROOT, MODEL_ROOT, DATA_ROOT, RUNS_ROOT, CHECKPOINT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

run_identity = json.dumps({
    'target_phrase': TARGET_PHRASE,
    'model_name': MODEL_NAME,
    'language': LANGUAGE,
}, sort_keys=True).encode()
RUN_ID = hashlib.sha256(run_identity).hexdigest()[:12]
RUN_ROOT = RUNS_ROOT / f'{MODEL_NAME}-{RUN_ID}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Keep package and Hugging Face caches in disposable local scratch.
os.environ['PIP_CACHE_DIR'] = str(CACHE_ROOT / 'pip')
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HF_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'datasets')
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
for key in ('PIP_CACHE_DIR', 'HF_HOME', 'HF_HUB_CACHE', 'HF_DATASETS_CACHE'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.part')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')
    os.replace(temporary, path)


def remote_size(url):
    try:
        response = requests.head(
            url, allow_redirects=True, timeout=(30, 120),
            headers={'Accept-Encoding': 'identity'})
        response.raise_for_status()
        value = response.headers.get('content-length')
        return int(value) if value and value.isdigit() else None
    except Exception as exc:
        print(f'  HEAD unavailable ({type(exc).__name__}); download validation will use format checks')
        return None


def file_valid(path, expected_size=None, minimum_size=1, expected_sha256=None, validator=None):
    path = Path(path)
    try:
        if not path.is_file() or path.stat().st_size < minimum_size:
            return False
        if expected_size is not None and path.stat().st_size != expected_size:
            return False
        if expected_sha256 is not None and sha256_file(path) != expected_sha256:
            return False
        if validator is not None and not validator(path):
            return False
        return True
    except Exception:
        return False


def atomic_download(url, destination, *, minimum_size=1, expected_size=None,
                    expected_sha256=None, validator=None, description=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    part = destination.with_suffix(destination.suffix + '.part')
    metadata_path = destination.with_suffix(destination.suffix + '.download.json')
    metadata = {}
    if metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text())
        except Exception:
            metadata = {}
    expected_size = expected_size or remote_size(url) or metadata.get('size')

    if file_valid(destination, expected_size, minimum_size, expected_sha256, validator):
        print(f'  cached + verified: {description or destination.name} ({destination.stat().st_size/1e6:.1f} MB)')
        return destination
    if destination.exists():
        if (expected_size is None or destination.stat().st_size < expected_size) and not part.exists():
            os.replace(destination, part)
        else:
            destination.unlink()
    if file_valid(part, expected_size, minimum_size, expected_sha256, validator):
        os.replace(part, destination)
        atomic_json(metadata_path, {'url': url, 'size': destination.stat().st_size})
        print(f'  recovered complete .part: {description or destination.name}')
        return destination
    if part.exists() and expected_size is not None and part.stat().st_size > expected_size:
        part.unlink()

    for attempt in range(1, 7):
        start = part.stat().st_size if part.exists() else 0
        headers = {'Accept-Encoding': 'identity'}
        if start:
            headers['Range'] = f'bytes={start}-'
        try:
            with requests.get(url, stream=True, allow_redirects=True,
                              timeout=(30, 300), headers=headers) as response:
                if start and response.status_code == 416 and file_valid(
                        part, expected_size, minimum_size, expected_sha256, validator):
                    os.replace(part, destination)
                    return destination
                response.raise_for_status()
                if start and response.status_code != 206:
                    print('  server ignored Range; restarting this file safely')
                    start = 0
                content_range = response.headers.get('content-range', '')
                if start and not content_range.startswith(f'bytes {start}-'):
                    raise RuntimeError(f'unexpected Content-Range: {content_range!r}')
                mode = 'ab' if start else 'wb'
                response_length = response.headers.get('content-length')
                if response_length and response_length.isdigit():
                    response_total = start + int(response_length)
                    if expected_size is None:
                        expected_size = response_total
                total = expected_size or None
                with open(part, mode) as handle, tqdm(
                    total=total, initial=start, unit='B', unit_scale=True,
                    unit_divisor=1024, desc=description or destination.name
                ) as progress:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))
                    handle.flush()
                    os.fsync(handle.fileno())
            if not file_valid(part, expected_size, minimum_size, expected_sha256, validator):
                actual = part.stat().st_size if part.exists() else 0
                raise RuntimeError(f'integrity check failed: {actual} bytes, expected {expected_size}')
            os.replace(part, destination)
            atomic_json(metadata_path, {'url': url, 'size': destination.stat().st_size})
            print(f'  downloaded + verified: {description or destination.name}')
            return destination
        except Exception as exc:
            if attempt == 6:
                raise
            delay = 15 * attempt
            print(f'  attempt {attempt}/6 interrupted: {type(exc).__name__}: {str(exc)[:140]}')
            print(f'  keeping {part.name}; retrying in {delay}s')
            time.sleep(delay)


VOICE_CATALOG_PATH = CACHE_ROOT / 'piper-voices.json'
atomic_download(
    'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/voices.json',
    VOICE_CATALOG_PATH, minimum_size=100_000,
    validator=lambda p: isinstance(json.loads(p.read_text()), dict),
    description='Piper voice catalog')
voices_catalog = json.loads(VOICE_CATALOG_PATH.read_text())
valid_codes = sorted({v['language']['code'] for v in voices_catalog.values()} | {'en_US'})
assert LANGUAGE in valid_codes, f'No Piper voices for {LANGUAGE}. Valid: {", ".join(valid_codes)}'

atomic_json(RUN_ROOT / 'run.json', {
    'target_phrase': TARGET_PHRASE,
    'model_name': MODEL_NAME,
    'language': LANGUAGE,
    'run_id': RUN_ID,
    'last_runtime_phase': RUNTIME_PHASE,
})
print(f'Runtime phase:        {RUNTIME_PHASE}')
print('Storage budget:      Drive ~7–9 GB peak; CPU scratch ~10–12 GB peak')
print(f'Local cache/raw:      {CACHE_ROOT}, {LOCAL_DATA_ROOT}')
print(f'Persistent workspace: {DRIVE_ROOT}')
print(f'This model run:       {RUN_ROOT}')
print('Step 0 complete. Drive outputs survive runtime termination; local raw .part downloads do not.')
_STEP_0_OK = True


## 0M. Optional one-time migration from the legacy raw-Drive layout

Run this cell **only for the current over-quota workspace**, immediately after Step 0 on a CPU runtime. It reads the already-verified ACAV file from Drive and writes only the required 1.9008 GB train/validation subsets to local scratch. After both subsets pass exact shape, dtype, and size checks, it pauses while you delete the legacy raw Drive files and empty Trash. Step 5 then copies the compact subsets back to Drive.


In [ ]:
assert '_STEP_0_OK' in dir(), 'Run Step 0 on a CPU runtime first!'
require_phase('CPU', 'Step 0M migration')

import os, shutil
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

EXPECTED_ACAV_SOURCE_SIZE = 17_280_000_128
EXPECTED_ACAV_SOURCE_SHAPE = (5_625_000, 16, 96)
EXPECTED_ACAV_TRAIN_SHAPE = (562_500, 16, 96)
EXPECTED_ACAV_VAL_SHAPE = (900_000, 96)
EXPECTED_ACAV_TRAIN_SIZE = 1_728_000_128
EXPECTED_ACAV_VAL_SIZE = 172_800_128
legacy_candidates = [
    DATA_ROOT / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    DATA_ROOT / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy.part',
]
local_train = LOCAL_DATA_ROOT / 'migration-acav-train.npy'
local_val = LOCAL_DATA_ROOT / 'migration-acav-val.npy'


def valid_npy(path, expected_shape, expected_size):
    try:
        array = np.load(path, mmap_mode='r')
        return (path.stat().st_size == expected_size and
                tuple(array.shape) == tuple(expected_shape) and
                array.dtype == np.dtype('<f2') and array.flags.c_contiguous)
    except Exception:
        return False


def atomic_save_local(destination, array):
    part = destination.with_suffix(destination.suffix + '.part')
    part.unlink(missing_ok=True)
    with open(part, 'wb') as handle:
        np.save(handle, array)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(part, destination)


legacy_drive_acav = None
source = None
for candidate in legacy_candidates:
    if candidate.exists() and candidate.stat().st_size == EXPECTED_ACAV_SOURCE_SIZE:
        try:
            check = np.load(candidate, mmap_mode='r')
            if tuple(check.shape) == EXPECTED_ACAV_SOURCE_SHAPE and check.dtype == np.dtype('<f2') and check.flags.c_contiguous:
                legacy_drive_acav = candidate
                source = check
                break
            del check
        except Exception:
            pass
if legacy_drive_acav is None:
    raise RuntimeError(
        'No complete valid legacy ACAV .npy or .npy.part was found in Drive. '
        'Skip Step 0M; Step 5 will fetch only the required 1.9008 GB byte ranges.')

required_local = EXPECTED_ACAV_TRAIN_SIZE + EXPECTED_ACAV_VAL_SIZE + 500_000_000
free = shutil.disk_usage('/content').free
if free < required_local:
    raise RuntimeError(
        f'Need at least {required_local/1e9:.1f} GB local free to stage compact ACAV subsets; '
        f'only {free/1e9:.1f} GB is available.')
print(f'Validated legacy ACAV source: {legacy_drive_acav}')

if not valid_npy(local_train, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE):
    local_train.unlink(missing_ok=True)
    print(f'Creating local train subset: {EXPECTED_ACAV_TRAIN_SHAPE}')
    atomic_save_local(local_train, source[:EXPECTED_ACAV_TRAIN_SHAPE[0]])
if not valid_npy(local_val, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE):
    local_val.unlink(missing_ok=True)
    first_val_row = EXPECTED_ACAV_TRAIN_SHAPE[0]
    val_rows = EXPECTED_ACAV_VAL_SHAPE[0] // EXPECTED_ACAV_SOURCE_SHAPE[1]
    print(f'Creating local validation subset: {EXPECTED_ACAV_VAL_SHAPE}')
    validation = source[first_val_row:first_val_row + val_rows]
    atomic_save_local(local_val, validation.reshape(EXPECTED_ACAV_VAL_SHAPE))
del source
assert valid_npy(local_train, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE)
assert valid_npy(local_val, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE)
print(f'Validated local train subset: {local_train} ({local_train.stat().st_size/1e9:.3f} GB)')
print(f'Validated local validation subset: {local_val} ({local_val.stat().st_size/1e9:.3f} GB)')

print("""
NOW, WITHOUT DISCONNECTING THIS CPU RUNTIME:
1. Open drive.google.com → My Drive → HOUSEVOICE_Wakeword.
2. Delete these legacy raw items if present:
   datasets/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
   datasets/openwakeword_features_ACAV100M_2000_hrs_16bit.npy.part
   datasets/openwakeword_features_ACAV100M_2000_hrs_16bit.npy.download.json
   datasets/fma_small.zip
   datasets/fma_small.zip.part
   datasets/fma_small.zip.download.json
   datasets/fma/  (raw extracted MP3 tree)
   cache/         (old persistent pip/Hugging Face caches)
3. Do NOT delete datasets/acav_*_subset.npy, datasets/fma_wav_16khz_1500,
   datasets/mit_rirs_16khz, models, sources, runs, or checkpoints.
4. Empty Google Drive Trash so quota is actually reclaimed.
""")
confirmation = input('After deleting the raw items and EMPTYING TRASH, type DRIVE RAW DELETED: ').strip().upper()
assert confirmation == 'DRIVE RAW DELETED', 'Migration paused; keep this runtime and do not continue until Drive quota is reclaimed.'
print('Compact ACAV subsets remain validated in local scratch. Continue with CPU Steps 1–8A in order.')
_STEP_0M_OK = True


## 1. Install phase-appropriate pinned dependencies

CPU mode installs the complete preparation/export stack. GPU mode installs only YAML support and verifies the untouched Colab Torch/Torchaudio/CUDA stack, minimizing paid/limited GPU time. Both modes reuse the persistent Drive pip cache.


In [ ]:
assert '_STEP_0_OK' in dir(), 'Run Step 0 (Phase + Drive) first!'

import importlib, importlib.metadata, subprocess, sys
from pathlib import Path

EXPECTED = {'python': (3, 12), 'numpy': '2.0.2', 'torch': '2.11.0', 'torchaudio': '2.11.0'}
if sys.version_info[:2] != EXPECTED['python']:
    raise RuntimeError(
        f'Unsupported Python {sys.version.split()[0]}. Select Colab runtime version 2026.07. '
        'Do not force-install into another Python version.'
    )

import numpy as np, torch, torchaudio
base = lambda value: value.split('+', 1)[0]
base_errors = []
if np.__version__ != EXPECTED['numpy']: base_errors.append(f'numpy={np.__version__}')
if base(torch.__version__) != EXPECTED['torch']: base_errors.append(f'torch={torch.__version__}')
if base(torchaudio.__version__) != EXPECTED['torchaudio']: base_errors.append(f'torchaudio={torchaudio.__version__}')
if base_errors:
    raise RuntimeError('Wrong/modified Colab base image: ' + ', '.join(base_errors) + '. Select runtime 2026.07.')
if RUNTIME_PHASE == 'GPU' and not torch.cuda.is_available():
    raise RuntimeError('GPU phase requires CUDA. Attach a T4 or L4, then rerun Steps 0 and 1.')
if RUNTIME_PHASE == 'CPU' and torch.cuda.is_available():
    raise RuntimeError('CPU phase must use Hardware accelerator: None to preserve GPU quota.')
print('Pinned base:', sys.version.split()[0], np.__version__, torch.__version__, torchaudio.__version__)
print('Runtime phase:', RUNTIME_PHASE)
print('Compute:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

constraints = Path('/content/housevoice-constraints.txt')
constraints.write_text('numpy==2.0.2\ntorch==2.11.0\ntorchaudio==2.11.0\n')
if RUNTIME_PHASE == 'CPU':
    subprocess.run([
        'apt-get', 'install', '-y', '-qq', 'cmake', 'espeak-ng', 'espeak-ng-data',
        'libespeak-ng-dev', 'libsndfile1', 'pkg-config', 'build-essential', 'ffmpeg', 'unzip'
    ], check=True)
    packages = [
        'piper-tts==1.3.0', 'webrtcvad-wheels==2.0.14', 'mutagen==1.47.0',
        'torchinfo==1.8.0', 'torchmetrics==1.8.2', 'PyYAML==6.0.2', 'tqdm==4.67.1',
        'datasets==2.21.0', 'soundfile==0.13.1', 'audiomentations==0.33.0',
        'torch-audiomentations==0.12.0', 'pronouncing==0.2.0', 'onnxruntime==1.22.1',
        'onnx==1.18.0', 'speechbrain==1.1.2', 'acoustics==0.2.6', 'scipy==1.15.3',
        'requests==2.32.4', 'huggingface-hub==0.34.4', 'transformers==4.57.6',
        'deep-phonemizer==0.0.19'
    ]
else:
    # Step 8B uses only NumPy, Torch, and YAML. NumPy/Torch come from the pinned
    # Colab image and are constrained above; do not spend GPU time installing
    # the CPU preparation/export dependency set.
    packages = ['PyYAML==6.0.2']
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--constraint', str(constraints), *packages
], check=True)

importlib.invalidate_caches()
import numpy as np, torch, torchaudio
assert np.__version__ == EXPECTED['numpy'], np.__version__
assert base(torch.__version__) == EXPECTED['torch'], torch.__version__
assert base(torchaudio.__version__) == EXPECTED['torchaudio'], torchaudio.__version__
if RUNTIME_PHASE == 'GPU':
    assert torch.cuda.is_available(), 'CUDA disappeared after dependency installation'
else:
    from piper import PiperVoice, SynthesisConfig
    from piper.phonemize_espeak import EspeakPhonemizer
print(f'Pinned {RUNTIME_PHASE} dependencies installed; Torch/Torchaudio unchanged.')
_STEP_1_OK = True


## 2. CPU — Restore pinned sources and voice models (~1–3 min)

Exact source archives and Piper voices are downloaded once to Drive with atomic resume/integrity checks. Each CPU runtime extracts clean disposable source trees under `/content`, then reapplies compatibility adjustments.


In [ ]:
assert '_STEP_1_OK' in dir(), 'Run Step 1 (Install) first!'
require_phase('CPU', 'Step 2')

import importlib, inspect, os, shutil, subprocess, sys, tarfile
from pathlib import Path

PSG_COMMIT = '2971426a55072f7d22fec416ca7800df8bd23207'
OWW_COMMIT = '368c03716d1e92591906a84949bc477f3a834455'
PSG_DIR = '/content/piper-sample-generator'
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'


def valid_source_archive(path, required_suffix):
    try:
        with tarfile.open(path, 'r:gz') as archive:
            names = archive.getnames()
            return len(names) > 1 and any(name.endswith(required_suffix) for name in names)
    except Exception:
        return False


def restore_source(name, repo, commit, destination, required_suffix):
    archive_path = SOURCE_ROOT / f'{name}-{commit}.tar.gz'
    url = f'https://github.com/{repo}/archive/{commit}.tar.gz'
    atomic_download(
        url, archive_path, minimum_size=50_000,
        validator=lambda p: valid_source_archive(p, required_suffix),
        description=f'{name} source {commit[:8]}')
    destination = Path(destination)
    marker = destination / '.housevoice_source_commit'
    if marker.exists() and marker.read_text().strip() == commit and (destination / required_suffix).exists():
        print(f'  local source ready: {name}')
        return
    temporary = Path(f'{destination}.extracting')
    shutil.rmtree(temporary, ignore_errors=True)
    temporary.mkdir(parents=True)
    with tarfile.open(archive_path, 'r:gz') as archive:
        archive.extractall(temporary, filter='data')
    roots = [item for item in temporary.iterdir() if item.is_dir()]
    if len(roots) != 1 or not (roots[0] / required_suffix).exists():
        raise RuntimeError(f'Unexpected {name} source archive layout')
    shutil.rmtree(destination, ignore_errors=True)
    shutil.move(str(roots[0]), destination)
    shutil.rmtree(temporary, ignore_errors=True)
    marker.write_text(commit + '\n')
    print(f'  restored clean source: {name} {commit}')


restore_source(
    'piper-sample-generator', 'rhasspy/piper-sample-generator', PSG_COMMIT,
    PSG_DIR, 'piper_sample_generator/__main__.py')
restore_source(
    'openwakeword', 'dscripka/openWakeWord', OWW_COMMIT,
    OWW_DIR, 'openwakeword/train.py')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--no-deps', '--editable', OWW_DIR
], check=True)

for source_dir in (PSG_DIR, OWW_DIR):
    if source_dir not in sys.path:
        sys.path.insert(0, source_dir)
for module_name in list(sys.modules):
    if module_name.startswith('openwakeword'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import openwakeword
from piper_sample_generator.__main__ import generate_samples as _piper_generate_samples
from piper_train.vits import commons as _piper_commons
assert openwakeword.__file__ is not None, 'OpenWakeWord loaded as a namespace package'
assert Path(OWW_TRAIN).exists(), OWW_TRAIN
print(f'  OpenWakeWord: {openwakeword.__file__}')
print(f'  Piper generator: {_piper_generate_samples.__module__}')

PIPER_MODELS = MODEL_ROOT / 'piper'
PIPER_MODELS.mkdir(parents=True, exist_ok=True)
if LANGUAGE == 'en_US':
    PIPER_MODEL = PIPER_MODELS / 'en_US-libritts_r-medium.pt'
    atomic_download(
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
        PIPER_MODEL, expected_size=204_089_915, minimum_size=100_000_000,
        description='Piper LibriTTS voice')
    source_config = Path(PSG_DIR) / 'models' / 'en_US-libritts_r-medium.pt.json'
    persistent_config = Path(str(PIPER_MODEL) + '.json')
    if not persistent_config.exists():
        if not source_config.exists():
            raise RuntimeError(f'Piper model config missing: {source_config}')
        shutil.copy2(source_config, persistent_config)
    json.loads(persistent_config.read_text())
    VOICE_MODELS = str(PIPER_MODEL)
else:
    VOICES_BASE_URL = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/'
    quality_rank = {'high': 0, 'medium': 1, 'low': 2, 'x_low': 3}
    matches = {k: v for k, v in voices_catalog.items() if v['language']['code'] == LANGUAGE}
    if not matches:
        raise ValueError(f'No Piper voices for {LANGUAGE}')
    sorted_voices = sorted(matches.values(), key=lambda value: quality_rank.get(value['quality'], 9))
    onnx_files = []
    language_dir = PIPER_MODELS / LANGUAGE
    language_dir.mkdir(parents=True, exist_ok=True)
    for voice in sorted_voices:
        for relative_path, info in voice['files'].items():
            if not (relative_path.endswith('.onnx') or relative_path.endswith('.onnx.json')):
                continue
            local = language_dir / Path(relative_path).name
            atomic_download(
                VOICES_BASE_URL + relative_path, local,
                expected_size=info['size_bytes'], minimum_size=100,
                description=local.name)
            if relative_path.endswith('.onnx') and not relative_path.endswith('.onnx.json'):
                onnx_files.append(str(local))
    assert onnx_files, f'No ONNX voices for {LANGUAGE}'
    VOICE_MODELS = onnx_files

# OpenWakeWord imports generate_samples.py from the configured Piper source.
shim_code = f"""\
import sys
sys.path.insert(0, {PSG_DIR!r})
from piper_sample_generator.__main__ import generate_samples as _gen_pt
from piper_sample_generator.__main__ import generate_samples_onnx as _gen_onnx
_DEFAULT_MODEL = {VOICE_MODELS!r}
def generate_samples(*, model=_DEFAULT_MODEL, text, max_samples, output_dir,
                     file_names=None, noise_scales=None, noise_scale_ws=None,
                     length_scales=None, **kwargs):
    shared=dict(text=text,max_samples=max_samples,output_dir=output_dir)
    if file_names is not None: shared["file_names"]=file_names
    if noise_scales: shared["noise_scales"]=noise_scales
    if noise_scale_ws: shared["noise_scale_ws"]=noise_scale_ws
    if length_scales: shared["length_scales"]=length_scales
    if isinstance(model,list) or (isinstance(model,str) and model.endswith(".onnx")):
        return _gen_onnx(model=model if isinstance(model,list) else [model],**shared)
    return _gen_pt(model=model,batch_size=kwargs.pop("batch_size",50),**shared,**kwargs)
"""
for shim_path in (Path(PSG_DIR) / 'generate_samples.py', Path(OWW_DIR) / 'openwakeword' / 'generate_samples.py'):
    shim_path.write_text(shim_code)

import torch_audiomentations
_ta_root = Path(inspect.getfile(torch_audiomentations)).resolve().parent
assert not any('set_audio_backend(' in candidate.read_text(errors='ignore')
               for candidate in _ta_root.rglob('*.py')), _ta_root

# Adapt only the disposable pinned OpenWakeWord checkout to Torchaudio 2.11.
DATA_PY = Path(OWW_DIR) / 'openwakeword' / 'data.py'
data_text = DATA_PY.read_text()
marker = '# -- HOUSEVOICE SoundFile compatibility for Torchaudio 2.11 --'
if marker not in data_text:
    helper = r"""
# -- HOUSEVOICE SoundFile compatibility for Torchaudio 2.11 --
import types as _types
import soundfile as _soundfile

def _housevoice_audio_info(file_path, *args, **kwargs):
    info=_soundfile.info(str(file_path))
    return _types.SimpleNamespace(
        num_frames=info.frames,sample_rate=info.samplerate,
        num_channels=info.channels,bits_per_sample=16,encoding='PCM_S')

def _housevoice_load_audio(file_path, frame_offset=0, num_frames=-1, *args, **kwargs):
    frames=-1 if num_frames is None or num_frames < 0 else num_frames
    audio,sample_rate=_soundfile.read(
        str(file_path),start=frame_offset,frames=frames,
        dtype='float32',always_2d=True)
    return torch.from_numpy(audio.T.copy()),sample_rate
"""
    data_text = data_text.replace('import torchaudio\n', 'import torchaudio\n' + helper + '\n', 1)
data_text = data_text.replace('torchaudio.info(', '_housevoice_audio_info(')
data_text = data_text.replace('torchaudio.load(', '_housevoice_load_audio(')
DATA_PY.write_text(data_text)
assert 'torchaudio.info(' not in data_text and 'torchaudio.load(' not in data_text

train_path = Path(OWW_TRAIN)
train_text = train_path.read_text().replace(
    'val_predictions = self.model(x_val)\n',
    'val_predictions = self.model(x_val.float())\n')
train_path.write_text(train_text)

print(f'Persistent voice model(s): {VOICE_MODELS}')
print('Pinned sources, voice models, and compatibility adjustments are ready.')
_STEP_2_OK = True


## 3. CPU — Dependency pre-flight

Hard-fails on a wrong CPU runtime, changed source commit, missing import, or accidental GPU allocation before slow preparation begins.


In [ ]:
assert '_STEP_2_OK' in dir(), 'Run Step 2 (Sources + voices) first!'
require_phase('CPU', 'Step 3')

import importlib, importlib.metadata, inspect, os, sys
from pathlib import Path
import numpy as np, torch, torchaudio
errors = []
if sys.version_info[:2] != (3, 12): errors.append(f'Python {sys.version.split()[0]} (expected 3.12.x)')
if np.__version__ != '2.0.2': errors.append(f'NumPy {np.__version__} (expected 2.0.2)')
if torch.__version__.split('+')[0] != '2.11.0': errors.append(f'Torch {torch.__version__} (expected 2.11.0)')
if torchaudio.__version__.split('+')[0] != '2.11.0': errors.append(f'Torchaudio {torchaudio.__version__} (expected 2.11.0)')
if torch.cuda.is_available(): errors.append('CPU pre-flight unexpectedly has CUDA; use Hardware accelerator: None')

for dist, expected in {
    'piper-tts': '1.3.0', 'audiomentations': '0.33.0', 'torch-audiomentations': '0.12.0',
    'onnxruntime': '1.22.1', 'onnx': '1.18.0', 'speechbrain': '1.1.2',
    'transformers': '4.57.6', 'huggingface-hub': '0.34.4', 'deep-phonemizer': '0.0.19'
}.items():
    try:
        actual = importlib.metadata.version(dist)
        print(f'  {dist}: {actual}')
        if actual != expected: errors.append(f'{dist} {actual} (expected {expected})')
    except Exception as exc:
        errors.append(f'{dist}: {exc}')

for directory, commit in ((PSG_DIR, PSG_COMMIT), (OWW_DIR, OWW_COMMIT)):
    marker_path = Path(directory) / '.housevoice_source_commit'
    actual = marker_path.read_text().strip() if marker_path.exists() else 'missing'
    print(f'  {Path(directory).name}: {actual}')
    if actual != commit: errors.append(f'{directory} at {actual}, expected {commit}')

for module in [
    'torchinfo', 'torchmetrics', 'scipy', 'tqdm', 'yaml', 'mutagen', 'pronouncing',
    'audiomentations', 'torch_audiomentations', 'speechbrain', 'acoustics', 'onnx',
    'onnxruntime', 'soundfile', 'requests', 'huggingface_hub', 'piper',
    'piper.phonemize_espeak', 'piper_sample_generator', 'piper_train.vits.commons',
    'transformers', 'openwakeword', 'dp.model'
]:
    try: importlib.import_module(module)
    except Exception as exc: errors.append(f'import {module}: {type(exc).__name__}: {exc}')

try:
    from transformers.utils.hub import is_offline_mode
    assert callable(is_offline_mode)
    import torchmetrics as _torchmetrics_regression
    print('  regression PASS: transformers + torchmetrics imports')
except Exception as exc:
    errors.append(f'transformers/torchmetrics regression: {type(exc).__name__}: {exc}')

try:
    import tempfile
    import soundfile as sf
    from speechbrain.dataio.dataio import read_audio as sb_read_audio
    from speechbrain.utils.torch_audio_backend import check_torchaudio_backend
    check_torchaudio_backend()
    with tempfile.NamedTemporaryFile(suffix='.wav') as test_wav:
        sf.write(test_wav.name, np.zeros(1600, dtype=np.float32), 16000)
        loaded = sb_read_audio(test_wav.name)
        assert tuple(loaded.shape) == (1600,), loaded.shape
    print('  regression PASS: speechbrain audio on torchaudio 2.11')
except Exception as exc:
    errors.append(f'speechbrain/torchaudio regression: {type(exc).__name__}: {exc}')

try:
    from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
    from openwakeword.utils import compute_features_from_generator, AudioFeatures
    source = Path(inspect.getfile(importlib.import_module('openwakeword.data'))).read_text()
    assert 'torchaudio.info(' not in source and 'torchaudio.load(' not in source
    print('  regression PASS: OpenWakeWord uses SoundFile compatibility')
except Exception as exc:
    errors.append(f'OpenWakeWord training imports: {type(exc).__name__}: {exc}')

required_files = {
    'voice model': VOICE_MODELS if isinstance(VOICE_MODELS, str) else VOICE_MODELS[0],
    'sample shim': f'{PSG_DIR}/generate_samples.py',
    'train.py': OWW_TRAIN,
}
if isinstance(VOICE_MODELS, str) and VOICE_MODELS.endswith('.pt'):
    required_files['voice model config'] = f'{VOICE_MODELS}.json'
for label, required_path in required_files.items():
    if not os.path.exists(required_path) or os.path.getsize(required_path) == 0:
        errors.append(f'{label} missing: {required_path}')
print('  compute: CPU only (GPU intentionally not allocated)')
print('  Drive workspace:', DRIVE_ROOT)
if errors: raise RuntimeError('Pre-flight failed:\n- ' + '\n- '.join(errors))
print('\nDependency pre-flight PASSED. Run the speech gate next.')
_STEP_3_OK = True


## 4. CPU — HOUSEVOICE speech smoke test

Generates one fresh Piper utterance on CPU. Listen and type `HOUSEVOICE`; this preserves the proven speech gate without consuming GPU quota.


In [ ]:
assert '_STEP_3_OK' in dir(), 'Run Step 3 (Dependency pre-flight) first!'
require_phase('CPU', 'Step 4')
import importlib.util, shutil, wave
from pathlib import Path
import numpy as np
from IPython.display import Audio, display

sample_dir=Path('/content/housevoice-smoke-test')
shutil.rmtree(sample_dir,ignore_errors=True); sample_dir.mkdir(parents=True)
spec=importlib.util.spec_from_file_location('housevoice_generate_samples',f'{PSG_DIR}/generate_samples.py')
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
module.generate_samples(model=VOICE_MODELS,text='house voice',max_samples=1,output_dir=str(sample_dir),batch_size=1,max_speakers=32)
sample_path=sample_dir/'0.wav'
assert sample_path.exists(),'Piper did not create the HOUSEVOICE sample'
with wave.open(str(sample_path),'rb') as wav:
    channels,width,rate,frames=wav.getnchannels(),wav.getsampwidth(),wav.getframerate(),wav.getnframes()
    pcm=np.frombuffer(wav.readframes(frames),dtype=np.int16)
duration=frames/rate; rms=float(np.sqrt(np.mean(pcm.astype(np.float64)**2))) if pcm.size else 0.0
assert channels==1 and width==2,(channels,width)
assert rate==22050,f'Expected 22050 Hz, got {rate}'
assert duration>=0.25,f'Sample too short: {duration:.3f}s'
assert rms>=100,f'Sample appears silent: RMS={rms:.1f}'
print(f'Generated {sample_path}: {duration:.2f}s, {rate} Hz, mono, RMS={rms:.1f}')
display(Audio(filename=str(sample_path),autoplay=False))
confirmation=input('Listen above. If it clearly says HOUSEVOICE, type HOUSEVOICE: ').strip().upper()
assert confirmation=='HOUSEVOICE','Speech gate not confirmed. Stop and troubleshoot before training.'
print('HOUSEVOICE speech gate PASSED. Persistent downloads may proceed or resume.')
_STEP_SAMPLE_OK=True


## 5. CPU — Build persistent prepared datasets from local raw scratch

Raw ACAV/FMA data never becomes a durable Drive asset. ACAV downloads only the byte ranges needed for the configured train/validation subsets. FMA downloads its archive to `/content`, converts tracks directly into persistent 16-kHz WAVs, then deletes the archive. Existing prepared subsets/WAVs are verified and skipped.


In [ ]:
assert '_STEP_SAMPLE_OK' in dir(), 'Run Step 4 (HOUSEVOICE speech smoke test) first!'
require_phase('CPU', 'Step 5')

import io, json, math, os, shutil, subprocess, time, zipfile
from pathlib import Path

import datasets
import numpy as np
import onnx
import scipy.io.wavfile as wavfile
import soundfile as sf
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
local_free = shutil.disk_usage('/content').free
if local_free < 2_000_000_000:
    raise RuntimeError(f'Step 5 needs at least 2 GB initial local headroom; only {local_free/1e9:.1f} GB is available.')
print(f'  local scratch available: {local_free/1e9:.1f} GB')


def valid_onnx(path):
    try:
        model_proto = onnx.load(str(path), load_external_data=False)
        onnx.checker.check_model(model_proto)
        return True
    except Exception:
        return False


def valid_wav(path, sample_rate=None, channels=None):
    try:
        info = sf.info(str(path))
        return (info.frames > 0 and
                (sample_rate is None or info.samplerate == sample_rate) and
                (channels is None or info.channels == channels))
    except Exception:
        return False


def valid_subset(path, expected_shape, expected_size):
    try:
        array = np.load(path, mmap_mode='r')
        return (path.stat().st_size == expected_size and
                tuple(array.shape) == tuple(expected_shape) and
                array.dtype == np.dtype('<f2') and array.flags.c_contiguous)
    except Exception:
        return False


def promote_valid_part(destination, expected_shape, expected_size):
    destination = Path(destination)
    part = destination.with_suffix(destination.suffix + '.part')
    if not destination.exists() and valid_subset(part, expected_shape, expected_size):
        os.replace(part, destination)
    elif part.exists() and not valid_subset(part, expected_shape, expected_size):
        part.unlink()


def atomic_save_npy(destination, array):
    destination = Path(destination)
    part = destination.with_suffix(destination.suffix + '.part')
    with open(part, 'wb') as handle:
        np.save(handle, array)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(part, destination)


# Shared OpenWakeWord models remain small persistent Drive assets.
shared_model_dir = MODEL_ROOT / 'openwakeword-v0.5.1'
shared_model_dir.mkdir(parents=True, exist_ok=True)
local_model_dir = Path(OWW_DIR) / 'openwakeword' / 'resources' / 'models'
local_model_dir.mkdir(parents=True, exist_ok=True)
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
shared_model_sizes = {
    'embedding_model.onnx': 1_326_578,
    'embedding_model.tflite': 1_330_312,
    'melspectrogram.onnx': 1_087_958,
    'melspectrogram.tflite': 1_092_516,
}
for filename in shared_model_sizes:
    persistent = shared_model_dir / filename
    validator = valid_onnx if filename.endswith('.onnx') else None
    atomic_download(
        f'{base_url}/{filename}', persistent,
        expected_size=shared_model_sizes[filename], minimum_size=1_000,
        validator=validator, description=f'OpenWakeWord {filename}')
    local = local_model_dir / filename
    if local.exists() or local.is_symlink():
        local.unlink()
    local.symlink_to(persistent)

# MIT RIR converted WAVs persist; Hugging Face's raw cache is local scratch.
MIT_DIR = DATA_ROOT / 'mit_rirs_16khz'
MIT_DIR.mkdir(parents=True, exist_ok=True)
for part in MIT_DIR.glob('*.wav.part'):
    output = part.with_name(part.name.removesuffix('.part'))
    if valid_wav(part) and not valid_wav(output):
        os.replace(part, output)
    else:
        part.unlink()
valid_rirs = [path for path in MIT_DIR.glob('*.wav') if valid_wav(path)]
for path in MIT_DIR.glob('*.wav'):
    if path not in valid_rirs:
        path.unlink()
if len(valid_rirs) < 250:
    for attempt in range(1, 7):
        try:
            snapshot_download(
                repo_id='davidscripka/MIT_environmental_impulse_responses',
                repo_type='dataset', etag_timeout=120, max_workers=4,
                cache_dir=os.environ['HF_HUB_CACHE'])
            break
        except Exception as exc:
            if attempt == 6: raise
            time.sleep(15 * attempt)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses', split='train',
        streaming=False, cache_dir=os.environ['HF_DATASETS_CACHE'])
    for index, row in enumerate(tqdm(rir_dataset, desc='MIT RIR conversion')):
        output = MIT_DIR / f'{index:04d}.wav'
        if valid_wav(output):
            continue
        temporary = Path('/content') / f'housevoice-rir-{index:04d}.wav'
        audio = row['audio']
        sample_rate = audio['sampling_rate']
        array = np.asarray(audio['array'])
        if sample_rate != 16000:
            from scipy.signal import resample_poly
            array = resample_poly(array, 16000, sample_rate)
        array = (array * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(temporary, 16000, array)
        if not valid_wav(temporary):
            raise RuntimeError(f'Invalid converted RIR: {temporary}')
        drive_part = output.with_suffix('.wav.part')
        shutil.copyfile(temporary, drive_part)
        os.replace(drive_part, output)
        temporary.unlink(missing_ok=True)
valid_rirs = [path for path in MIT_DIR.glob('*.wav') if valid_wav(path)]
assert len(valid_rirs) >= 250, f'Only {len(valid_rirs)} valid MIT RIR WAVs'
print(f'  MIT RIRs verified: {len(valid_rirs)}')

# ACAV: retain only the configured 10% train and 1% validation subsets.
# The NPY payload is row-major, so HTTP byte ranges avoid the other 89%.
ACAV_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
EXPECTED_ACAV_SOURCE_SIZE = 17_280_000_128
EXPECTED_ACAV_SOURCE_SHAPE = (5_625_000, 16, 96)
EXPECTED_ACAV_TRAIN_SHAPE = (562_500, 16, 96)
EXPECTED_ACAV_VAL_SHAPE = (900_000, 96)
EXPECTED_ACAV_TRAIN_SIZE = 1_728_000_128
EXPECTED_ACAV_VAL_SIZE = 172_800_128
ACAV_TRAIN = DATA_ROOT / 'acav_train_subset.npy'
ACAV_VAL = DATA_ROOT / 'acav_val_subset.npy'
LOCAL_MIGRATION_TRAIN = LOCAL_DATA_ROOT / 'migration-acav-train.npy'
LOCAL_MIGRATION_VAL = LOCAL_DATA_ROOT / 'migration-acav-val.npy'
promote_valid_part(ACAV_TRAIN, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE)
promote_valid_part(ACAV_VAL, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE)


def remote_npy_layout(url):
    with requests.get(
        url, stream=True, allow_redirects=True, timeout=(30, 120),
        headers={'Range': 'bytes=0-65535', 'Accept-Encoding': 'identity'}
    ) as response:
        response.raise_for_status()
        if response.status_code != 206:
            raise RuntimeError(
                f'Server did not honor the ACAV header range (HTTP {response.status_code}); refusing a full download')
        content_range = response.headers.get('content-range', '')
        if not content_range.endswith(f'/{EXPECTED_ACAV_SOURCE_SIZE}'):
            raise RuntimeError(f'Unexpected ACAV source size in Content-Range: {content_range!r}')
        prefix = bytearray()
        for chunk in response.iter_content(chunk_size=16 * 1024):
            prefix.extend(chunk)
            if len(prefix) >= 65536:
                break
    handle = io.BytesIO(prefix)
    version = np.lib.format.read_magic(handle)
    if version == (1, 0):
        shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(handle)
    elif version == (2, 0):
        shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(handle)
    else:
        raise RuntimeError(f'Unsupported remote NPY format version: {version}')
    if fortran_order:
        raise RuntimeError('ACAV NPY unexpectedly uses Fortran order; byte-range slicing is unsafe')
    return tuple(shape), np.dtype(dtype), handle.tell()


def download_range(url, destination, first_byte, last_byte, description):
    destination = Path(destination)
    part = destination.with_suffix(destination.suffix + '.part')
    expected = last_byte - first_byte + 1
    if destination.exists() and destination.stat().st_size == expected:
        return destination
    if destination.exists(): destination.unlink()
    if part.exists() and part.stat().st_size > expected: part.unlink()
    for attempt in range(1, 7):
        resume = part.stat().st_size if part.exists() else 0
        headers = {
            'Range': f'bytes={first_byte + resume}-{last_byte}',
            'Accept-Encoding': 'identity',
        }
        try:
            with requests.get(url, stream=True, allow_redirects=True,
                              timeout=(30, 300), headers=headers) as response:
                if response.status_code != 206:
                    raise RuntimeError(
                        f'Server did not honor ACAV byte range (HTTP {response.status_code}); refusing full 17.3 GB download')
                content_range = response.headers.get('content-range', '')
                if not content_range.startswith(f'bytes {first_byte + resume}-'):
                    raise RuntimeError(f'Unexpected Content-Range: {content_range!r}')
                if not content_range.endswith(f'/{EXPECTED_ACAV_SOURCE_SIZE}'):
                    raise RuntimeError(f'Unexpected ACAV source size in Content-Range: {content_range!r}')
                with open(part, 'ab') as handle, tqdm(
                    total=expected, initial=resume, unit='B', unit_scale=True,
                    unit_divisor=1024, desc=description
                ) as progress:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))
                    handle.flush(); os.fsync(handle.fileno())
            if part.stat().st_size != expected:
                raise RuntimeError(f'Range size {part.stat().st_size}, expected {expected}')
            os.replace(part, destination)
            return destination
        except Exception as exc:
            if attempt == 6: raise
            print(f'  ACAV range attempt {attempt}/6: {type(exc).__name__}: {str(exc)[:120]}')
            time.sleep(15 * attempt)



def copy_local_subset(source, destination, expected_shape, expected_size, description):
    if not valid_subset(source, expected_shape, expected_size):
        return False
    if valid_subset(destination, expected_shape, expected_size):
        return True
    destination.unlink(missing_ok=True)
    part = destination.with_suffix(destination.suffix + '.part')
    part.unlink(missing_ok=True)
    with open(source, 'rb') as input_handle, open(part, 'wb') as output_handle, tqdm(
        total=expected_size, unit='B', unit_scale=True, unit_divisor=1024, desc=description
    ) as progress:
        for chunk in iter(lambda: input_handle.read(8 * 1024 * 1024), b''):
            output_handle.write(chunk)
            progress.update(len(chunk))
        output_handle.flush()
        os.fsync(output_handle.fileno())
    if not valid_subset(part, expected_shape, expected_size):
        raise RuntimeError(f'Persistent copy validation failed: {description}')
    os.replace(part, destination)
    return True


# Prefer compact subsets staged by Step 0M, then remove local migration copies
# only after both persistent Drive files pass exact validation.
copy_local_subset(
    LOCAL_MIGRATION_TRAIN, ACAV_TRAIN,
    EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE,
    'Persist migrated ACAV train subset')
copy_local_subset(
    LOCAL_MIGRATION_VAL, ACAV_VAL,
    EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE,
    'Persist migrated ACAV validation subset')

if not (valid_subset(ACAV_TRAIN, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE) and
        valid_subset(ACAV_VAL, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE)):
    full_shape, acav_dtype, header_bytes = remote_npy_layout(ACAV_URL)
    if full_shape != EXPECTED_ACAV_SOURCE_SHAPE or acav_dtype != np.dtype('<f2'):
        raise RuntimeError(f'Unexpected ACAV layout: shape={full_shape}, dtype={acav_dtype}')
    n_train = EXPECTED_ACAV_TRAIN_SHAPE[0]
    n_val_rows = EXPECTED_ACAV_VAL_SHAPE[0] // EXPECTED_ACAV_SOURCE_SHAPE[1]
    row_shape = full_shape[1:]
    row_bytes = math.prod(row_shape) * acav_dtype.itemsize
    need_train = not valid_subset(ACAV_TRAIN, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE)
    need_val = not valid_subset(ACAV_VAL, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE)
    first_row = 0 if need_train else n_train
    row_count = (n_train if need_train else 0) + (n_val_rows if need_val else 0)
    payload = LOCAL_DATA_ROOT / f'acav-rows-{first_row}-{row_count}.bin'
    first_byte = header_bytes + first_row * row_bytes
    last_byte = first_byte + row_count * row_bytes - 1
    download_range(ACAV_URL, payload, first_byte, last_byte, 'ACAV required rows')
    selected = np.memmap(
        payload, dtype=acav_dtype, mode='r',
        shape=(row_count, *row_shape), order='C')
    cursor = 0
    if need_train:
        ACAV_TRAIN.unlink(missing_ok=True)
        atomic_save_npy(ACAV_TRAIN, selected[:n_train])
        cursor += n_train
    if need_val:
        ACAV_VAL.unlink(missing_ok=True)
        validation = selected[cursor:cursor + n_val_rows]
        atomic_save_npy(ACAV_VAL, validation.reshape(EXPECTED_ACAV_VAL_SHAPE))
    del selected
    payload.unlink(missing_ok=True)
    payload.with_suffix(payload.suffix + '.part').unlink(missing_ok=True)
assert valid_subset(ACAV_TRAIN, EXPECTED_ACAV_TRAIN_SHAPE, EXPECTED_ACAV_TRAIN_SIZE), 'Invalid persistent ACAV training subset'
assert valid_subset(ACAV_VAL, EXPECTED_ACAV_VAL_SHAPE, EXPECTED_ACAV_VAL_SIZE), 'Invalid persistent ACAV validation subset'
LOCAL_MIGRATION_TRAIN.unlink(missing_ok=True)
LOCAL_MIGRATION_VAL.unlink(missing_ok=True)
LOCAL_MIGRATION_TRAIN.with_suffix('.npy.part').unlink(missing_ok=True)
LOCAL_MIGRATION_VAL.with_suffix('.npy.part').unlink(missing_ok=True)
print(f'  ACAV train persisted: {np.load(ACAV_TRAIN, mmap_mode="r").shape}, {ACAV_TRAIN.stat().st_size/1e9:.3f} GB')
print(f'  ACAV val persisted:   {np.load(ACAV_VAL, mmap_mode="r").shape}, {ACAV_VAL.stat().st_size/1e9:.3f} GB')

# FMA: raw ZIP is local scratch. Convert directly to persistent WAVs without
# storing the archive or extracted MP3 tree in Drive.
FMA_WAV_DIR = DATA_ROOT / 'fma_wav_16khz_1500'
FMA_WAV_DIR.mkdir(parents=True, exist_ok=True)
FMA_URL = 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'
FMA_ZIP = LOCAL_DATA_ROOT / 'fma_small.zip'


def valid_16k_mono_wav(path):
    return valid_wav(path, sample_rate=16000, channels=1)


for part in FMA_WAV_DIR.glob('*.wav.part'):
    output = part.with_name(part.name.removesuffix('.part'))
    if valid_16k_mono_wav(part) and not valid_16k_mono_wav(output):
        os.replace(part, output)
    else:
        part.unlink()
for candidate in FMA_WAV_DIR.glob('*.wav'):
    if not valid_16k_mono_wav(candidate):
        candidate.unlink()
valid_fma_wavs = {path.name for path in FMA_WAV_DIR.glob('*.wav') if valid_16k_mono_wav(path)}
if len(valid_fma_wavs) < 1_500:
    fma_scratch_free = shutil.disk_usage('/content').free
    if fma_scratch_free < 9_000_000_000:
        raise RuntimeError(
            f'FMA needs 9 GB local scratch; only {fma_scratch_free/1e9:.1f} GB remains. '
            'Restart the CPU runtime after ACAV subsets are safely in Drive, then rerun Steps 0–5.')
    def valid_fma_zip(path):
        try:
            with zipfile.ZipFile(path) as archive:
                members = [item for item in archive.infolist() if not item.is_dir()]
                return len(members) >= 8_000 and archive.testzip() is None
        except Exception:
            return False
    atomic_download(
        FMA_URL, FMA_ZIP, expected_size=7_679_594_875,
        minimum_size=7_000_000_000, validator=valid_fma_zip,
        description='FMA small local archive')
    with zipfile.ZipFile(FMA_ZIP) as archive:
        mp3_members = sorted(
            (item for item in archive.infolist() if item.filename.lower().endswith('.mp3')),
            key=lambda item: item.filename)
        for index, member in enumerate(tqdm(mp3_members, desc='FMA → persistent 16 kHz WAV', unit='track')):
            if len(valid_fma_wavs) >= 1_500:
                break
            track_name = f'{index:05d}.wav'
            output = FMA_WAV_DIR / track_name
            if track_name in valid_fma_wavs and valid_16k_mono_wav(output):
                continue
            local_mp3 = LOCAL_DATA_ROOT / 'fma-current.mp3'
            local_wav = LOCAL_DATA_ROOT / 'fma-current.wav'
            local_mp3.unlink(missing_ok=True); local_wav.unlink(missing_ok=True)
            with archive.open(member) as source, open(local_mp3, 'wb') as target:
                shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
            result = subprocess.run([
                'ffmpeg', '-y', '-loglevel', 'error', '-i', str(local_mp3),
                '-ac', '1', '-ar', '16000', str(local_wav)
            ])
            if result.returncode != 0 or not valid_16k_mono_wav(local_wav):
                print(f'  skipping unreadable FMA track: {member.filename}')
                continue
            drive_part = output.with_suffix('.wav.part')
            shutil.copyfile(local_wav, drive_part)
            if not valid_16k_mono_wav(drive_part):
                drive_part.unlink(missing_ok=True)
                raise RuntimeError(f'Persistent FMA WAV verification failed: {track_name}')
            os.replace(drive_part, output)
            valid_fma_wavs.add(track_name)
    FMA_ZIP.unlink(missing_ok=True)
    FMA_ZIP.with_suffix(FMA_ZIP.suffix + '.part').unlink(missing_ok=True)
    FMA_ZIP.with_suffix(FMA_ZIP.suffix + '.download.json').unlink(missing_ok=True)
    (LOCAL_DATA_ROOT / 'fma-current.mp3').unlink(missing_ok=True)
    (LOCAL_DATA_ROOT / 'fma-current.wav').unlink(missing_ok=True)
valid_fma_wavs = [path for path in FMA_WAV_DIR.glob('*.wav') if valid_16k_mono_wav(path)]
assert len(valid_fma_wavs) >= 1_500, f'Only {len(valid_fma_wavs)} valid persistent FMA WAVs'
print(f'  FMA WAVs persisted: {len(valid_fma_wavs)}, {sum(p.stat().st_size for p in valid_fma_wavs)/1e9:.2f} GB')

# Warn about legacy raw Drive assets; migration instructions remove them via
# Drive web UI and empty Trash so quota is actually reclaimed.
legacy_raw = [
    DATA_ROOT / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    DATA_ROOT / 'fma_small.zip',
    DATA_ROOT / 'fma_small.zip.part',
    DATA_ROOT / 'fma',
]
remaining_legacy = [str(path) for path in legacy_raw if path.exists()]
if remaining_legacy:
    print('WARNING: legacy raw Drive assets still consume quota and are no longer used:')
    for path in remaining_legacy: print(' ', path)
    print('Delete them in drive.google.com and empty Trash after verifying prepared outputs above.')

DOWNLOAD_MANIFEST = RUN_ROOT / 'step5-prepared-data.json'
atomic_json(DOWNLOAD_MANIFEST, {
    'architecture': 'local_raw_drive_prepared_v1',
    'acav_train_path': str(ACAV_TRAIN),
    'acav_train_size': ACAV_TRAIN.stat().st_size,
    'acav_val_path': str(ACAV_VAL),
    'acav_val_size': ACAV_VAL.stat().st_size,
    'fma_wav_path': str(FMA_WAV_DIR),
    'fma_wav_count': len(valid_fma_wavs),
    'mit_rir_count': len(valid_rirs),
    'shared_models': [str(path) for path in sorted(shared_model_dir.iterdir()) if path.is_file()],
})
print(f'Step 5 prepared-data manifest: {DOWNLOAD_MANIFEST}')
_STEP_4_OK = True


## 6. CPU — Verify prepared persistent data

This step performs no raw-data download or conversion. It verifies that the compact ACAV subsets and 1,500 FMA WAVs needed by later steps are present in Drive.


In [ ]:
assert '_STEP_4_OK' in dir(), 'Run Step 5 (Local raw → persistent prepared data) first!'
require_phase('CPU', 'Step 6')

from pathlib import Path
import numpy as np
import soundfile as sf

ACAV_TRAIN = DATA_ROOT / 'acav_train_subset.npy'
ACAV_VAL = DATA_ROOT / 'acav_val_subset.npy'
FMA_WAV_DIR = DATA_ROOT / 'fma_wav_16khz_1500'


def valid_subset(path, expected_shape, expected_size):
    try:
        array = np.load(path, mmap_mode='r')
        return (path.stat().st_size == expected_size and
                tuple(array.shape) == tuple(expected_shape) and
                array.dtype == np.dtype('<f2') and array.flags.c_contiguous)
    except Exception:
        return False


def valid_16k_mono_wav(path):
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.samplerate == 16000 and info.channels == 1
    except Exception:
        return False


assert valid_subset(ACAV_TRAIN, (562_500, 16, 96), 1_728_000_128), f'Invalid/missing {ACAV_TRAIN}'
assert valid_subset(ACAV_VAL, (900_000, 96), 172_800_128), f'Invalid/missing {ACAV_VAL}'
valid_fma_wavs = [path for path in FMA_WAV_DIR.glob('*.wav') if valid_16k_mono_wav(path)]
assert len(valid_fma_wavs) >= 1_500, f'Only {len(valid_fma_wavs)} valid FMA WAVs'
print(f'  ACAV train: {np.load(ACAV_TRAIN, mmap_mode="r").shape}, {ACAV_TRAIN.stat().st_size/1e9:.2f} GB')
print(f'  ACAV val:   {np.load(ACAV_VAL, mmap_mode="r").shape}, {ACAV_VAL.stat().st_size/1e9:.2f} GB')
print(f'  FMA WAVs:   {len(valid_fma_wavs)}, {sum(p.stat().st_size for p in valid_fma_wavs)/1e9:.2f} GB')
_STEP_5_OK = True


## 7. CPU — Build config and generate/resume TTS clips

Piper TTS generation and resampling are CPU-compatible. They may take time, but valid clips persist and are reused without holding a GPU allocation.


In [ ]:
assert '_STEP_5_OK' in dir(), 'Run Step 6 (Prepare audio) first!'
require_phase('CPU', 'Step 7')

import glob, hashlib, inspect, json, os, sys, time, yaml
from pathlib import Path

import soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm

CONFIG_PATH = RUN_ROOT / 'my_model.yaml'
MODEL_OUTPUT = RUN_ROOT / 'output'
FEATURE_ROOT = MODEL_OUTPUT / MODEL_NAME
config = {
    'target_phrase': TARGET_PHRASE,
    'model_name': MODEL_NAME,
    'custom_negative_phrases': [],
    'n_samples': 2000,
    'n_samples_val': 1000,
    'tts_batch_size': 50,
    'piper_sample_generator_path': PSG_DIR,
    'augmentation_rounds': 1,
    'augmentation_batch_size': 16,
    'steps': 20000,
    'max_negative_weight': 1500,
    'target_accuracy': 0.7,
    'target_recall': 0.5,
    'target_false_positives_per_hour': 0.5,
    'batch_size': 128,
    'learning_rate': 1e-4,
    'model_type': 'dnn',
    'layer_dim': 128,
    'layer_size': 128,
    'n_blocks': 1,
    'model_input_shape': [16, 96],
    'n_classes': 1,
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },
    'background_paths': [str(FMA_WAV_DIR)],
    'background_paths_duplication_rate': [1],
    'rir_paths': [str(MIT_DIR)],
    'false_positive_validation_data_path': str(ACAV_VAL),
    'feature_data_files': {'ACAV100M_sample': str(ACAV_TRAIN)},
    'output_dir': str(MODEL_OUTPUT),
    'tflite_export': True,
    'onnx_export': True,
    'positive_clips_train_dir': str(FEATURE_ROOT / 'positive_train'),
    'positive_clips_test_dir': str(FEATURE_ROOT / 'positive_test'),
    'negative_clips_train_dir': str(FEATURE_ROOT / 'negative_train'),
    'negative_clips_test_dir': str(FEATURE_ROOT / 'negative_test'),
    'feature_save_dir': str(FEATURE_ROOT),
}
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.dump(config, sort_keys=False))
config_fingerprint = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
print(f'  persistent config: {CONFIG_PATH}')
print(f'  config fingerprint: {config_fingerprint}')

# PyTorch 2.x compatibility for deep-phonemizer.
import dp.model.model as dp_model
dp_path = Path(inspect.getfile(dp_model))
dp_text = dp_path.read_text().replace(
    'torch.load(checkpoint_path, map_location=device)',
    'torch.load(checkpoint_path, map_location=device, weights_only=False)')
dp_path.write_text(dp_text)

with open(CONFIG_PATH) as handle:
    cfg = yaml.safe_load(handle)
clip_dirs = {
    'positive_train': (Path(cfg['positive_clips_train_dir']), int(cfg['n_samples'] * 0.75)),
    'positive_test': (Path(cfg['positive_clips_test_dir']), int(cfg['n_samples_val'] * 0.75)),
    'negative_train': (Path(cfg['negative_clips_train_dir']), int(cfg['n_samples'] * 0.75)),
    'negative_test': (Path(cfg['negative_clips_test_dir']), int(cfg['n_samples_val'] * 0.75)),
}


def valid_clip(path):
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.channels == 1 and info.samplerate in (16000, 22050)
    except Exception:
        return False


def clip_files(directory):
    return [path for path in directory.glob('*.wav') if not path.name.endswith('.part.wav')]


clips_changed = False
for name, (directory, expected) in clip_dirs.items():
    directory.mkdir(parents=True, exist_ok=True)
    # Recover or discard an interrupted atomic resample without counting its
    # temporary file as an additional training clip.
    for partial in directory.glob('*.part.wav'):
        original = partial.with_name(partial.name.removesuffix('.part.wav') + '.wav')
        if valid_clip(partial) and not valid_clip(original):
            os.replace(partial, original)
        else:
            partial.unlink()
        clips_changed = True
    for candidate in clip_files(directory):
        if not valid_clip(candidate):
            candidate.unlink()
            clips_changed = True
    count = sum(1 for candidate in clip_files(directory) if valid_clip(candidate))
    print(f'  {"PASS" if count >= expected else "MISS"} {name}: {count} valid clips (expect >= {expected})')

if not all(sum(1 for candidate in clip_files(directory) if valid_clip(candidate)) >= expected
           for directory, expected in clip_dirs.values()):
    before = {name: len(clip_files(directory)) for name, (directory, _) in clip_dirs.items()}
    subprocess_result = __import__('subprocess').run([
        sys.executable, OWW_TRAIN, '--training_config', str(CONFIG_PATH), '--generate_clips'
    ])
    if subprocess_result.returncode != 0:
        raise RuntimeError('OpenWakeWord clip generation failed; rerun this step to resume cached directories')
    after = {name: len(clip_files(directory)) for name, (directory, _) in clip_dirs.items()}
    clips_changed = clips_changed or before != after

for name, (directory, expected) in clip_dirs.items():
    valid = [candidate for candidate in clip_files(directory) if valid_clip(candidate)]
    assert len(valid) >= expected, f'{name} has only {len(valid)} valid clips; expected >= {expected}'

for name, (directory, _) in clip_dirs.items():
    for path in tqdm(clip_files(directory), desc=f'resample {name}', leave=False):
        info = sf.info(str(path))
        if info.samplerate == 16000:
            continue
        data, sample_rate = sf.read(path)
        converted = resample_poly(data.astype('float32'), 16000, sample_rate)
        temporary = path.with_suffix('.part.wav')
        sf.write(temporary, converted, 16000)
        if not valid_clip(temporary) or sf.info(str(temporary)).samplerate != 16000:
            raise RuntimeError(f'Resample validation failed: {path}')
        os.replace(temporary, path)
        clips_changed = True

feature_marker = FEATURE_ROOT / '.feature-config.json'
feature_state = {}
if feature_marker.exists():
    try: feature_state = json.loads(feature_marker.read_text())
    except Exception: feature_state = {}
if clips_changed or feature_state.get('config_fingerprint') != config_fingerprint:
    removed = 0
    for feature_path in FEATURE_ROOT.glob('*_features_*.npy'):
        feature_path.unlink()
        removed += 1
    feature_state = {
        'config_fingerprint': config_fingerprint,
        'generation_id': time.time_ns(),
    }
    if removed:
        print(f'  invalidated {removed} stale feature files because clips/config changed')
else:
    feature_state.setdefault('generation_id', time.time_ns())
atomic_json(feature_marker, feature_state)
print('Persistent TTS clips verified at 16 kHz.')
_STEP_6_OK = True


## 8A. CPU — Augment clips and extract persistent features

OpenWakeWord augmentation and embedding extraction support CPU execution. This is intentionally completed before requesting a GPU. The four validated feature arrays and generation fingerprint persist in Drive.


In [ ]:
assert '_STEP_6_OK' in dir(), 'Run Step 7 (CPU config + clips) first!'
require_phase('CPU', 'Step 8A')

import copy, hashlib, json, math, os, random, shutil, sys, time, yaml
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

with open(CONFIG_PATH) as handle:
    cfg = yaml.safe_load(handle)
FEAT = Path(cfg['feature_save_dir'])
needed = [
    'positive_features_train.npy', 'negative_features_train.npy',
    'positive_features_test.npy', 'negative_features_test.npy'
]


def valid_feature(path):
    try:
        array = np.load(path, mmap_mode='r')
        return array.size > 0 and array.ndim == 3 and tuple(array.shape[1:]) == (16, 96)
    except Exception:
        return False


features_regenerated = False
for name in needed:
    path = FEAT / name
    if path.exists() and not valid_feature(path):
        print(f'  removing incomplete feature file: {name}')
        path.unlink()
        features_regenerated = True
if not all(valid_feature(FEAT / name) for name in needed):
    features_regenerated = True
    result = __import__('subprocess').run([
        sys.executable, OWW_TRAIN, '--training_config', str(CONFIG_PATH), '--augment_clips'
    ])
    if result.returncode != 0:
        raise RuntimeError('Augmentation failed; rerun this cell to retry invalid/missing feature files')
for name in needed:
    assert valid_feature(FEAT / name), f'Invalid augmented feature file: {name}'
    print(f'  verified {name}: {(FEAT / name).stat().st_size/1e6:.1f} MB')
feature_marker = FEAT / '.feature-config.json'
feature_state = json.loads(feature_marker.read_text())
if features_regenerated:
    feature_state['generation_id'] = time.time_ns()
    atomic_json(feature_marker, feature_state)
feature_signature = {
    name: {
        'size': (FEAT / name).stat().st_size,
        'shape': list(np.load(FEAT / name, mmap_mode='r').shape),
        'dtype': str(np.load(FEAT / name, mmap_mode='r').dtype),
    } for name in needed
}
training_fingerprint = hashlib.sha256(json.dumps({
    'config': config_fingerprint,
    'feature_generation': feature_state['generation_id'],
    'features': feature_signature,
}, sort_keys=True).encode()).hexdigest()
print(f'  training fingerprint: {training_fingerprint}')


print('CPU feature preparation complete. Disconnect this runtime before allocating a GPU.')
_STEP_8A_OK = True


## 8B. GPU — Train or resume from the Drive checkpoint

Switch to a T4/L4 only now. On the new GPU runtime, set `RUNTIME_PHASE = 'GPU'`, rerun Steps 0 and 1, then jump directly here. The cell fails clearly without CUDA or complete CPU features.


In [ ]:
assert '_STEP_1_OK' in dir(), 'On the GPU runtime, rerun Steps 0 and 1 first!'
require_phase('GPU', 'Step 8B')

import copy, hashlib, json, math, os, random, shutil, sys, time, yaml
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

if not torch.cuda.is_available():
    raise RuntimeError('Step 8B requires a T4 or L4 GPU; CUDA is unavailable.')
CONFIG_PATH = RUN_ROOT / 'my_model.yaml'
if not CONFIG_PATH.exists():
    raise RuntimeError('Persistent config missing. Return to CPU and complete Steps 0–8A.')
with open(CONFIG_PATH) as handle:
    cfg = yaml.safe_load(handle)
config_fingerprint = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
FEAT = Path(cfg['feature_save_dir'])
needed = [
    'positive_features_train.npy', 'negative_features_train.npy',
    'positive_features_test.npy', 'negative_features_test.npy'
]

def valid_feature(path):
    try:
        array = np.load(path, mmap_mode='r')
        return array.size > 0 and array.ndim == 3 and tuple(array.shape[1:]) == (16, 96)
    except Exception:
        return False

for name in needed:
    if not valid_feature(FEAT / name):
        raise RuntimeError(f'Persistent feature missing/invalid: {name}. Return to CPU Step 8A.')
feature_marker = FEAT / '.feature-config.json'
if not feature_marker.exists():
    raise RuntimeError('Feature generation marker missing. Return to CPU Step 8A.')
feature_state = json.loads(feature_marker.read_text())
feature_signature = {
    name: {
        'size': (FEAT / name).stat().st_size,
        'shape': list(np.load(FEAT / name, mmap_mode='r').shape),
        'dtype': str(np.load(FEAT / name, mmap_mode='r').dtype),
    } for name in needed
}
training_fingerprint = hashlib.sha256(json.dumps({
    'config': config_fingerprint,
    'feature_generation': feature_state['generation_id'],
    'features': feature_signature,
}, sort_keys=True).encode()).hexdigest()
print(f'Persistent CPU features verified: {training_fingerprint}')

# Stage prepared arrays sequentially into local VM storage. This avoids random
# Google Drive reads starving the GPU during training. It copies only completed
# prepared files; it does not redownload or regenerate any dataset.
GPU_CACHE = Path('/content/housevoice-gpu-cache')
GPU_CACHE.mkdir(parents=True, exist_ok=True)
gpu_cache_marker = GPU_CACHE / 'training-fingerprint.txt'
if not gpu_cache_marker.exists() or gpu_cache_marker.read_text().strip() != training_fingerprint:
    for stale in GPU_CACHE.glob('*.npy*'):
        stale.unlink()
staging_sources = [FEAT / name for name in needed] + [
    Path(cfg['feature_data_files']['ACAV100M_sample']),
    Path(cfg['false_positive_validation_data_path']),
]
required_staging_bytes = sum(source.stat().st_size for source in staging_sources)
free_staging_bytes = shutil.disk_usage('/content').free
if free_staging_bytes < required_staging_bytes + 2_000_000_000:
    raise RuntimeError(
        f'Insufficient local GPU scratch space: need about {required_staging_bytes/1e9:.1f} GB '
        f'plus 2 GB headroom, have {free_staging_bytes/1e9:.1f} GB free.')

def stage_npy(source):
    source = Path(source)
    destination = GPU_CACHE / source.name
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        temporary = destination.with_suffix(destination.suffix + '.part')
        print(f'  staging {source.name}: {source.stat().st_size/1e9:.2f} GB')
        shutil.copyfile(source, temporary)
        if temporary.stat().st_size != source.stat().st_size:
            raise RuntimeError(f'Incomplete GPU scratch copy: {source.name}')
        os.replace(temporary, destination)
    try:
        staged = np.load(destination, mmap_mode='r')
        assert staged.size > 0
    except Exception as exc:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f'Invalid GPU scratch copy: {source.name}') from exc
    return destination

GPU_FEATURES = {name: stage_npy(FEAT / name) for name in needed}
GPU_ACAV_TRAIN = stage_npy(cfg['feature_data_files']['ACAV100M_sample'])
GPU_ACAV_VAL = stage_npy(cfg['false_positive_validation_data_path'])
gpu_cache_marker.write_text(training_fingerprint + '\n')
print('Prepared arrays staged locally; persistent Drive originals remain unchanged.')

DEVICE = 'cuda'
print(f'Device: {torch.cuda.get_device_name(0)}')
pos_train = torch.from_numpy(np.load(GPU_FEATURES['positive_features_train.npy']).astype(np.float32)).to(DEVICE)
neg_train = torch.from_numpy(np.load(GPU_FEATURES['negative_features_train.npy']).astype(np.float32)).to(DEVICE)
pos_test = torch.from_numpy(np.load(GPU_FEATURES['positive_features_test.npy']).astype(np.float32)).to(DEVICE)
neg_test = torch.from_numpy(np.load(GPU_FEATURES['negative_features_test.npy']).astype(np.float32)).to(DEVICE)
print(f'  pos_train={tuple(pos_train.shape)} neg_train={tuple(neg_train.shape)}')
print(f'  pos_test={tuple(pos_test.shape)} neg_test={tuple(neg_test.shape)}')

acav_train_np = np.load(GPU_ACAV_TRAIN, mmap_mode='r')
acav_val_np = np.load(GPU_ACAV_VAL, mmap_mode='r')
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
n_win_val = M - 16
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
VAL_BATCH = 4096
print(f'  ACAV train: {acav_train_np.shape}; validation listening time: {val_listen_hours:.2f} hr')


class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))


model = WakewordModel().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
TOTAL_STEPS = cfg['steps']
MAX_NEG_W = cfg['max_negative_weight']
TARGET_FP_PER_HR = cfg['target_false_positives_per_hour']
THRESH = 0.5
history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
best_models = []
CHECKPOINT_PATH = CHECKPOINT_ROOT / f'{MODEL_NAME}-{RUN_ID}-training.pt'
CHECKPOINT_EVERY = 250
resume_stage, resume_step, resume_optimizer = 1, 0, None
max_neg_w_now = MAX_NEG_W


def checkpoint_valid(path):
    try:
        value = torch.load(path, map_location='cpu', weights_only=False)
        return value.get('training_fingerprint') == training_fingerprint and 'model_state' in value
    except Exception:
        return False


if CHECKPOINT_PATH.exists():
    if checkpoint_valid(CHECKPOINT_PATH):
        saved = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        model.load_state_dict(saved['model_state'])
        history = saved['history']
        best_models = saved['best_models']
        resume_stage = saved['next_stage']
        resume_step = saved['next_step']
        resume_optimizer = saved.get('optimizer_state')
        max_neg_w_now = saved.get('max_neg_w', MAX_NEG_W)
        np.random.set_state(saved['numpy_rng'])
        random.setstate(saved['python_rng'])
        torch.set_rng_state(saved['torch_rng'])
        if torch.cuda.is_available() and saved.get('cuda_rng') is not None:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
        print(f'  RESUME checkpoint: stage={resume_stage}, next_step={resume_step}, best={len(best_models)}')
    else:
        incompatible = CHECKPOINT_PATH.with_name(
            f'{CHECKPOINT_PATH.stem}.invalid-{int(time.time())}{CHECKPOINT_PATH.suffix}')
        os.replace(CHECKPOINT_PATH, incompatible)
        print(f'  checkpoint did not match current data/config; preserved as {incompatible.name}')
        print('  starting training from stage 1')


def cpu_state_dict(state):
    return {key: value.detach().cpu().clone() for key, value in state.items()}


def save_checkpoint(next_stage, next_step, optimizer, max_negative_weight):
    payload = {
        'format_version': 1,
        'config_fingerprint': config_fingerprint,
        'training_fingerprint': training_fingerprint,
        'model_state': cpu_state_dict(model.state_dict()),
        'optimizer_state': optimizer.state_dict() if optimizer is not None else None,
        'history': history,
        'best_models': best_models,
        'next_stage': next_stage,
        'next_step': next_step,
        'max_neg_w': max_negative_weight,
        'numpy_rng': np.random.get_state(),
        'python_rng': random.getstate(),
        'torch_rng': torch.get_rng_state(),
        'cuda_rng': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'saved_at': time.time(),
    }
    local = Path('/content') / f'{MODEL_NAME}-{RUN_ID}-checkpoint.tmp'
    drive_part = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + '.part')
    torch.save(payload, local)
    shutil.copyfile(local, drive_part)
    os.replace(drive_part, CHECKPOINT_PATH)
    local.unlink(missing_ok=True)


B_POS, B_ANEG, B_ACAV = 32, 32, 64

def random_acav_window_batch(count):
    rows_count, time_count, feature_count = acav_train_np.shape
    rows = np.random.randint(0, rows_count, size=count)
    starts = np.random.randint(0, time_count - 16 + 1, size=count)
    output = np.empty((count, 16, feature_count), dtype=np.float32)
    for index, (row, start) in enumerate(zip(rows, starts)):
        output[index] = acav_train_np[row, start:start + 16, :].astype(np.float32)
    return output


def build_batch():
    positive_indexes = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
    negative_indexes = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
    positive = pos_train[positive_indexes]
    adversarial = neg_train[negative_indexes]
    acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
    inputs = torch.cat([positive, adversarial, acav], dim=0)
    labels = torch.cat([
        torch.ones(B_POS, device=DEVICE),
        torch.zeros(B_ANEG + B_ACAV, device=DEVICE)
    ])
    return inputs, labels


@torch.no_grad()
def validate(step_label):
    model.eval()
    positive_predictions = torch.sigmoid(model(pos_test)).squeeze(-1)
    negative_predictions = torch.sigmoid(model(neg_test)).squeeze(-1)
    recall = (positive_predictions >= THRESH).float().mean().item()
    accuracy = (((positive_predictions >= THRESH).sum() +
                 (negative_predictions < THRESH).sum()).item() /
                (pos_test.shape[0] + neg_test.shape[0]))
    false_positives = 0
    for index in range(0, n_win_val, VAL_BATCH):
        chunk = torch.from_numpy(acav_val_windows[index:index + VAL_BATCH]).to(DEVICE)
        false_positives += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
    fp_per_hour = false_positives / max(val_listen_hours, 1e-6)
    history['val_recall'].append(recall)
    history['val_accuracy'].append(accuracy)
    history['val_fp_per_hour'].append(fp_per_hour)
    history['val_n_fp'].append(false_positives)
    save = False
    if len(history['val_n_fp']) >= 3:
        fp_p50 = np.percentile(history['val_n_fp'], 50)
        recall_p5 = np.percentile(history['val_recall'], 5)
        if false_positives <= fp_p50 and recall >= recall_p5:
            best_models.append((
                cpu_state_dict(model.state_dict()),
                {'val_recall': recall, 'val_accuracy': accuracy,
                 'val_fp_per_hour': fp_per_hour, 'val_n_fp': false_positives}
            ))
            save = True
    print(f'  [{step_label}] recall={recall:.3f} acc={accuracy:.3f} fp/hr={fp_per_hour:.2f} saved={"+" if save else "-"}', flush=True)
    model.train()


def move_optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


def run_stage(stage_index, number_steps, learning_rate, max_negative_weight,
              validation_window_fraction=1.0, start_step=0, optimizer_state=None):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    if optimizer_state is not None:
        optimizer.load_state_dict(optimizer_state)
        move_optimizer_to_device(optimizer)
    weight_schedule = np.linspace(1.0, max_negative_weight, number_steps)
    validation_start = int(number_steps * (1.0 - validation_window_fraction))
    validation_steps = set(np.linspace(validation_start, number_steps - 1, 20).astype(int))
    warmup = max(1, number_steps // 5)
    hold = number_steps // 3
    accumulated = []
    last_checkpoint_step = start_step

    def optimize_accumulated():
        if not accumulated:
            return
        combined_logits = torch.cat([item[0] for item in accumulated])
        combined_labels = torch.cat([item[1] for item in accumulated])
        combined_weights = torch.cat([item[2] for item in accumulated])
        loss = (loss_fn(combined_logits, combined_labels) * combined_weights).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history['loss'].append(loss.item())
        accumulated.clear()

    print(f'\n=== Stage {stage_index}: resume at {start_step}/{number_steps}, lr={learning_rate}, max_neg_w={max_negative_weight} ===', flush=True)
    started = time.time()
    for step in range(start_step, number_steps):
        if step < warmup:
            current_lr = learning_rate * (step + 1) / warmup
        elif step < warmup + hold:
            current_lr = learning_rate
        else:
            decay_time = (step - warmup - hold) / max(1, number_steps - warmup - hold)
            current_lr = learning_rate * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_time)))
        for group in optimizer.param_groups:
            group['lr'] = current_lr
        inputs, labels = build_batch()
        logits = model(inputs).squeeze(-1)
        predictions = torch.sigmoid(logits)
        keep = ((labels == 0) & (predictions >= 0.001)) | ((labels == 1) & (predictions < 0.999))
        if keep.sum() > 0:
            kept_logits = logits[keep]
            kept_labels = labels[keep]
            negative_weight = weight_schedule[step]
            weights = torch.where(
                kept_labels > 0.5,
                torch.tensor(1.0, device=DEVICE),
                torch.tensor(negative_weight, device=DEVICE, dtype=torch.float32))
            accumulated.append((kept_logits, kept_labels, weights))
            if sum(item[0].shape[0] for item in accumulated) >= 128:
                optimize_accumulated()
        if step in validation_steps:
            elapsed = (time.time() - started) / 60
            validate(f'stage{stage_index} step {step}/{number_steps} ({elapsed:.1f}m)')
        checkpoint_due = step + 1 - last_checkpoint_step >= CHECKPOINT_EVERY
        if checkpoint_due and not accumulated:
            save_checkpoint(stage_index, step + 1, optimizer, max_negative_weight)
            last_checkpoint_step = step + 1
            print(f'  checkpoint: stage {stage_index}, next step {step + 1}')
    # Commit any final partial gradient batch before the stage-complete checkpoint.
    optimize_accumulated()
    save_checkpoint(stage_index, number_steps, optimizer, max_negative_weight)
    print(f'  checkpoint: stage {stage_index}, next step {number_steps}')
    return optimizer

stage_plan = [
    (1, TOTAL_STEPS, 1e-4, 0.25),
    (2, max(2000, TOTAL_STEPS // 10), 1e-5, 1.0),
    (3, max(2000, TOTAL_STEPS // 10), 1e-6, 1.0),
]
for stage_index, number_steps, learning_rate, validation_fraction in stage_plan:
    if stage_index < resume_stage:
        continue
    start = resume_step if stage_index == resume_stage else 0
    optimizer_state = resume_optimizer if stage_index == resume_stage else None
    run_stage(
        stage_index, number_steps, learning_rate, max_neg_w_now,
        validation_window_fraction=validation_fraction,
        start_step=start, optimizer_state=optimizer_state)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR:
        max_neg_w_now *= 2
        print(f'  adaptive negative weight for next stage: {max_neg_w_now}')
    save_checkpoint(stage_index + 1, 0, None, max_neg_w_now)
    resume_stage, resume_step, resume_optimizer = stage_index + 1, 0, None

print(f'\n=== Training complete. {len(best_models)} candidate checkpoints persisted. ===')
if history['val_fp_per_hour']:
    print(f'  best val_fp_per_hour: {min(history["val_fp_per_hour"]):.2f}')
    print(f'  best val_recall: {max(history["val_recall"]):.3f}')
    print(f'  best val_accuracy: {max(history["val_accuracy"]):.3f}')
print(f'  rolling checkpoint: {CHECKPOINT_PATH}')
_STEP_8B_OK = True


## 9. CPU — Reload the completed checkpoint and export

After GPU training completes, release the GPU. On a CPU runtime set `RUNTIME_PHASE = 'CPU'`, rerun Steps 0 and 1, then jump here. ONNX/TFLite conversion and verification do not require CUDA.


In [ ]:
assert '_STEP_1_OK' in dir(), 'On the CPU export runtime, rerun Steps 0 and 1 first!'
require_phase('CPU', 'Step 9')

import copy, hashlib, json, os, shutil, time, yaml
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

if torch.cuda.is_available():
    raise RuntimeError('Step 9 is CPU-compatible. Detach the GPU to preserve quota, then rerun Steps 0, 1, and 9.')
CONFIG_PATH = RUN_ROOT / 'my_model.yaml'
if not CONFIG_PATH.exists():
    raise RuntimeError('Persistent config missing. Complete the CPU preparation phase first.')
with open(CONFIG_PATH) as handle:
    cfg = yaml.safe_load(handle)
config_fingerprint = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
FEAT = Path(cfg['feature_save_dir'])
needed = [
    'positive_features_train.npy', 'negative_features_train.npy',
    'positive_features_test.npy', 'negative_features_test.npy'
]

def valid_feature(path):
    try:
        array = np.load(path, mmap_mode='r')
        return array.size > 0 and array.ndim == 3 and tuple(array.shape[1:]) == (16, 96)
    except Exception:
        return False

for name in needed:
    if not valid_feature(FEAT / name):
        raise RuntimeError(f'Persistent feature missing/invalid: {name}. Return to CPU Step 8A.')
feature_state = json.loads((FEAT / '.feature-config.json').read_text())
feature_signature = {
    name: {
        'size': (FEAT / name).stat().st_size,
        'shape': list(np.load(FEAT / name, mmap_mode='r').shape),
        'dtype': str(np.load(FEAT / name, mmap_mode='r').dtype),
    } for name in needed
}
training_fingerprint = hashlib.sha256(json.dumps({
    'config': config_fingerprint,
    'feature_generation': feature_state['generation_id'],
    'features': feature_signature,
}, sort_keys=True).encode()).hexdigest()

class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

CHECKPOINT_PATH = CHECKPOINT_ROOT / f'{MODEL_NAME}-{RUN_ID}-training.pt'
if not CHECKPOINT_PATH.exists():
    raise RuntimeError('Training checkpoint missing. Complete GPU Step 8B first.')
saved = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
if saved.get('training_fingerprint') != training_fingerprint:
    raise RuntimeError('Training checkpoint does not match the persistent config/features.')
if saved.get('next_stage', 0) < 4:
    raise RuntimeError(
        f'Training is incomplete (next_stage={saved.get("next_stage")}, '
        f'next_step={saved.get("next_step")}). Resume GPU Step 8B first.')
history = saved['history']
best_models = saved['best_models']
model = WakewordModel()
model.load_state_dict(saved['model_state'])
DEVICE = 'cpu'
pos_test = torch.from_numpy(np.load(FEAT / 'positive_features_test.npy').astype(np.float32))
print(f'Completed training checkpoint loaded on CPU: {CHECKPOINT_PATH}')

if not best_models:
    print('  no saved checkpoints — using current model state')
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s['val_accuracy'] for _, s in best_models]
    rcs  = [s['val_recall']   for _, s in best_models]
    fps  = [s['val_fp_per_hour'] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90  = np.percentile(rcs, 90)
    fp_p10  = np.percentile(fps, 10)
    print(f'  thresholds: acc≥{acc_p90:.3f}  recall≥{rc_p90:.3f}  fp/hr≤{fp_p10:.2f}')
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                 and sc['val_fp_per_hour'] <= fp_p10]
    print(f'  qualified: {len(qualified)} of {len(best_models)} checkpoints')
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))
        qualified = [sorted_models[0]]
        print('  no checkpoint qualified — fell back to single best')
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    qa = np.mean([s['val_accuracy'] for _, s in qualified])
    qr = np.mean([s['val_recall'] for _, s in qualified])
    qf = np.mean([s['val_fp_per_hour'] for _, s in qualified])
    print(f'  ensemble averaged: mean acc={qa:.3f}, recall={qr:.3f}, fp/hr={qf:.2f}')
model.load_state_dict(final_state)
model.eval()

# ── ONNX export (sigmoid baked in so APK runtime sees [0, 1]) ──────
out_path = str(RUN_ROOT / f"{cfg['model_name']}.onnx")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
onnx_work = f"/content/{cfg['model_name']}.onnx"
class WakewordExportable(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x): return torch.sigmoid(self.base(x))
export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)
torch.onnx.export(export_model, dummy, onnx_work,
                  input_names=['onnx::Flatten_0'], output_names=['output'],
                  dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                  opset_version=14, dynamo=False)
print(f'  ✓ built temporary ONNX ({os.path.getsize(onnx_work)/1e3:.0f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(onnx_work)
print(f'  input:  {sess.get_inputs()[0].name} {sess.get_inputs()[0].shape}')
print(f'  output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}')
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    onnx_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f'  positive test set: mean={onnx_scores.mean():.3f}, recall@0.5={(onnx_scores>=0.5).mean():.3f}')
if history['val_fp_per_hour']:
    print(f'  ACAV val FP/hour at 0.5: {history["val_fp_per_hour"][-1]:.2f}')
onnx_part = Path(out_path + '.part')
shutil.copyfile(onnx_work, onnx_part)
os.replace(onnx_part, out_path)
print(f'  ✓ persisted {out_path} ({os.path.getsize(out_path)/1e3:.0f} KB)')

# ── TFLite export (rebuild in Keras, transfer weights) ─────────────
tflite_path = None
if cfg.get('tflite_export', False):
    import tensorflow as tf
    with torch.no_grad():
        w1 = model.layer1.weight.cpu().numpy()
        b1 = model.layer1.bias.cpu().numpy()
        ln_gamma = model.layernorm1.weight.cpu().numpy()
        ln_beta  = model.layernorm1.bias.cpu().numpy()
        w2 = model.layer2.weight.cpu().numpy()
        b2 = model.layer2.bias.cpu().numpy()

    inputs = tf.keras.Input(shape=(16, 96), name='input')
    x = tf.keras.layers.Flatten()(inputs)
    x = tf.keras.layers.Dense(128, use_bias=True)(x)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Dense(1, activation='sigmoid', use_bias=True)(x)
    keras_model = tf.keras.Model(inputs, x)

    # Keras Dense: [in, out]; PyTorch Linear: [out, in] — transpose needed
    dense1 = keras_model.layers[2]
    dense1.set_weights([w1.T, b1])
    ln = keras_model.layers[3]
    ln.set_weights([ln_gamma, ln_beta])
    dense2 = keras_model.layers[5]
    dense2.set_weights([w2.T, b2])

    # Numerical equivalence check vs ONNX
    keras_scores = keras_model.predict(p_test_np, verbose=0).flatten()
    max_diff = np.max(np.abs(onnx_scores - keras_scores))
    print(f'  TFLite: Keras vs ONNX max abs diff = {max_diff:.2e}')

# ...... Verify ONNX export matches Keras model output .............................................
    assert max_diff < 1e-4, f'Keras/ONNX mismatch too large: {max_diff}'

    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    tflite_model = converter.convert()
    tflite_path = out_path.replace('.onnx', '.tflite')
    tflite_work = f"/content/{cfg['model_name']}.tflite"
    with open(tflite_work, 'wb') as f:
        f.write(tflite_model)
    print(f'  ✓ built temporary TFLite ({os.path.getsize(tflite_work)/1e3:.0f} KB)')

    # Verify TFLite interpreter output before replacing the persistent artifact.
    interpreter = tf.lite.Interpreter(model_path=tflite_work)
    interpreter.allocate_tensors()
    inp_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[0]
    tflite_scores = np.empty(len(p_test_np), dtype=np.float32)
    for i in range(len(p_test_np)):
        interpreter.set_tensor(inp_detail['index'], p_test_np[i:i+1])
        interpreter.invoke()
        tflite_scores[i] = interpreter.get_tensor(out_detail['index']).flatten()[0]
    tflite_diff = np.max(np.abs(onnx_scores - tflite_scores))
    print(f'  TFLite: interpreter vs ONNX max abs diff = {tflite_diff:.2e}')
    print(f'  TFLite: positive test mean={tflite_scores.mean():.3f}, recall@0.5={(tflite_scores>=0.5).mean():.3f}')
    tflite_part = Path(tflite_path + '.part')
    shutil.copyfile(tflite_work, tflite_part)
    os.replace(tflite_part, tflite_path)
    print(f'  ✓ persisted {tflite_path} ({os.path.getsize(tflite_path)/1e3:.0f} KB)')

# ── Persist metrics and artifact hashes ─────────────────────────────
artifacts = [out_path] + ([tflite_path] if tflite_path else [])
metrics_path = RUN_ROOT / 'training-metrics.json'
metrics = {
    'model_name': cfg['model_name'],
    'run_id': RUN_ID,
    'config_fingerprint': config_fingerprint,
    'completed_at_unix': time.time(),
    'best_val_fp_per_hour': min(history['val_fp_per_hour']) if history['val_fp_per_hour'] else None,
    'best_val_recall': max(history['val_recall']) if history['val_recall'] else None,
    'best_val_accuracy': max(history['val_accuracy']) if history['val_accuracy'] else None,
    'artifacts': {
        os.path.basename(path): {
            'path': str(path),
            'size': os.path.getsize(path),
            'sha256': sha256_file(path),
        } for path in artifacts
    },
}
atomic_json(metrics_path, metrics)
print(f'  persistent metrics: {metrics_path}')
for name, details in metrics['artifacts'].items():
    print(f'  {name} SHA-256: {details["sha256"]}')

# ── Download convenience copies ────────────────────────────────────
from google.colab import files
files.download(out_path)
if tflite_path:
    files.download(tflite_path)
print(f'\nDONE. Wake word "{cfg["model_name"]}" trained and persisted in {RUN_ROOT}.')
print('Drop the downloaded model(s) into your app:')
print(f'  .onnx:   {cfg["model_name"]}.onnx')
if tflite_path:
    print(f'  .tflite: {cfg["model_name"]}.tflite')
